In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")

In [2]:
def ablation_study(X_full, y, feature_groups, dataset_name):
    imputer = SimpleImputer(strategy="mean")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    model = RandomForestClassifier(n_estimators=200, random_state=42)

    print(f"\n{'='*65}")
    print(f"  Ablation Study: {dataset_name}")
    print(f"{'='*65}")
    print(f"  {'Feature Set':<35} {'Accuracy':>10} {'Std':>8} {'AUROC':>10} {'Std':>8}")
    print(f"  {'-'*61}")

    results = []

    for name, cols in feature_groups.items():
        available = [c for c in cols if c in X_full.columns]
        if not available:
            continue
        X = imputer.fit_transform(X_full[available])
        acc = cross_val_score(model, X, y, cv=skf, scoring="accuracy")
        auc = cross_val_score(model, X, y, cv=skf, scoring="roc_auc")
        marker = " <-- FULL" if name == "All Features (Full Model)" else ""
        print(f"  {name:<35} {acc.mean():>10.4f} {acc.std():>8.4f} "
              f"{auc.mean():>10.4f} {auc.std():>8.4f}{marker}")
        results.append({
            "Dataset": dataset_name,
            "Feature Set": name,
            "Accuracy": acc.mean(),
            "Accuracy Std": acc.std(),
            "AUROC": auc.mean(),
            "AUROC Std": auc.std()
        })

    return pd.DataFrame(results)

In [3]:
PRAAT_FEATURES   = ["jitter", "shimmer", "hnr"]
LIBROSA_FEATURES = ["pitch", "spectral_centroid", "zcr"]
MFCC_FEATURES    = [f"mfcc_{i}" for i in range(1, 14)]
ALL_FEATURES     = PRAAT_FEATURES + LIBROSA_FEATURES + MFCC_FEATURES

# For Parkinson's model specifically
PK_PRAAT    = ["jitter", "shimmer", "hnr", "nhr"]
PK_NONLINEAR = ["rpde", "dfa"]
PK_PITCH    = ["pitch"]
PK_ALL      = ["pitch", "hnr", "jitter", "shimmer", "nhr", "rpde", "dfa"]

general_groups = {
    "All Features (Full Model)":        ALL_FEATURES,
    "Without Praat Features":           LIBROSA_FEATURES + MFCC_FEATURES,
    "Without MFCCs":                    PRAAT_FEATURES + LIBROSA_FEATURES,
    "Without Librosa Features":         PRAAT_FEATURES + MFCC_FEATURES,
    "Praat Features Only":              PRAAT_FEATURES,
    "MFCCs Only":                       MFCC_FEATURES,
    "Librosa Only":                     LIBROSA_FEATURES,
}

pk_groups = {
    "All Features (Full Model)":        PK_ALL,
    "Without Nonlinear Features":       PK_PRAAT + PK_PITCH,
    "Without Praat Features":           PK_PITCH + PK_NONLINEAR,
    "Praat + Pitch Only":               PK_PRAAT + PK_PITCH,
    "Nonlinear Features Only":          PK_NONLINEAR,
    "Praat Features Only":              PK_PRAAT,
}

In [4]:
# Load datasets
df_pk = pd.read_csv("../data/parkinsons_tabular/parkinsons.csv").rename(columns={
    "MDVP:Fo(Hz)": "pitch", "HNR": "hnr",
    "MDVP:Jitter(Abs)": "jitter", "MDVP:Shimmer": "shimmer",
    "NHR": "nhr", "RPDE": "rpde", "DFA": "dfa"
})
df_resp  = pd.read_csv("../data/voice_features/voice_dataset_labeled_full.csv").dropna()
df_stress = pd.read_csv("../data/voice_features/stress_dataset.csv").dropna()
df_depr  = pd.read_csv("../data/voice_features/depression_dataset.csv").dropna()

# Run ablation
res_pk     = ablation_study(df_pk,     df_pk["status"],  pk_groups,      "Parkinson's Disease")
res_resp   = ablation_study(df_resp,   df_resp["label"],  general_groups, "Respiratory Abnormality")
res_stress = ablation_study(df_stress, df_stress["label"], general_groups, "Psychological Stress")
res_depr   = ablation_study(df_depr,   df_depr["label"],  general_groups, "Depression Indicators")


  Ablation Study: Parkinson's Disease
  Feature Set                           Accuracy      Std      AUROC      Std
  -------------------------------------------------------------
  All Features (Full Model)               0.8872   0.0384     0.9494   0.0197 <-- FULL
  Without Nonlinear Features              0.8923   0.0192     0.9395   0.0338
  Without Praat Features                  0.8718   0.0585     0.9387   0.0381
  Praat + Pitch Only                      0.8923   0.0192     0.9395   0.0338
  Nonlinear Features Only                 0.7538   0.0447     0.7354   0.0282
  Praat Features Only                     0.8308   0.0384     0.8626   0.0685

  Ablation Study: Respiratory Abnormality
  Feature Set                           Accuracy      Std      AUROC      Std
  -------------------------------------------------------------
  All Features (Full Model)               0.6934   0.0102     0.7573   0.0095 <-- FULL
  Without Praat Features                  0.6823   0.0092     0.7441  

In [5]:
import os
os.makedirs("../results", exist_ok=True)

all_ablation = pd.concat([res_pk, res_resp, res_stress, res_depr], ignore_index=True)
all_ablation.to_csv("../results/ablation_study.csv", index=False)
print("Saved: results/ablation_study.csv")

Saved: results/ablation_study.csv
